# IMD Data Cleaning — IE Datathon Iberdrola 2026

**Source:** HERMES — Catálogo de datos de la Red de Transporte de Interés General  
**Publisher:** Ministerio de Transportes y Movilidad Sostenible (MITMS) — Subdirección General de Planificación, Red Transeuropea y Logística (SGPRTL)  
**Datasets used:**
- `IMD total de vehículos por tramo` → https://hermes.tragsatec.es/catalogo/dataset/intensidad-media-diaria-imd-total-de-vehiculos-por-tramo-de-las-carreteras-de-la-red-de-transpo1  
- `IMD por tipo de vehículo por tramo` → https://hermes.tragsatec.es/catalogo/dataset/intensidad-media-diaria-imd-por-tipo-de-vehiculo-por-tramo-de-las-carreteras-de-la-red-de-trans1

**Licence:** Open data — Reutilización de la información del sector público (Ley 37/2007)  
**Scope:** Red de Carreteras del Estado (RCE) — autopistas, autovías y carreteras nacionales  
**Relevance for the Datathon:** IMD values by road segment are used to weight EV charging demand along interurban corridors, directly feeding the charging station placement optimisation model (Objective 1).

---
### Notebook structure
1. Dependencies & setup  
2. RAW layer — load KMZ files and persist without transformation  
3. PROCESSED layer — clean, standardise, filter, deduplicate, merge datasets  
   3.1 Inspect & parse KML Description | 3.2 Column rename | 3.3 CRS | 3.4 Types  
   3.5 Interurban filter | 3.6 Null imputation | **3.7 Deduplication** | 3.8 Derived cols  
   3.9 Merge | 3.10 Outlier flagging | 3.11 Final column selection  
4. Save processed outputs  
5. Output verification  
6. Summary & assumptions

## 1. Dependencies & Setup

In [1]:
# Install required libraries (uncomment if running on Google Colab)
# !pip install geopandas fiona lxml pyproj shapely --quiet

In [2]:
import os
import zipfile
import json
import re
from datetime import datetime

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import mapping

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded.')
print(f'geopandas version : {gpd.__version__}')
print(f'pandas version    : {pd.__version__}')

Libraries loaded.
geopandas version : 1.1.3
pandas version    : 2.2.3


In [3]:
# ── Directory & file path configuration
RAW_DIR       = 'Data/Raw'
PROCESSED_DIR = 'Data/Processed'

os.makedirs(RAW_DIR,       exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f'Raw dir       : {os.path.abspath(RAW_DIR)}')
print(f'Processed dir : {os.path.abspath(PROCESSED_DIR)}')

Raw dir       : c:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon\Data\Processed\IMD_cleaned\Data\Raw
Processed dir : c:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon\Data\Processed\IMD_cleaned\Data\Processed


### 1.1 Download IMD data from ArcGIS REST (JSON)

The data is fetched directly from the **Ministerio de Transportes** ArcGIS REST service  
(`mapas.fomento.gob.es/arcgis2/rest/services/Hermes/3_USO_DE_LA_RED/MapServer`).  
Using `f=json` avoids the KMZ/KML parsing complexity entirely.

**Layer used:** `1025` — IMD total de vehículos por tramo (Red de Carreteras del Estado)  

The ArcGIS REST API returns a maximum of **1 000 features per request** (`resultRecordCount` cap).  
The download function therefore **paginates automatically** using `resultOffset` until all  
features are retrieved (`exceededTransferLimit = false`).

In [4]:
import urllib.request
import urllib.parse
import json
import time

# ── ArcGIS REST base endpoint
BASE_URL    = 'https://mapas.fomento.gob.es/arcgis2/rest/services/Hermes/3_USO_DE_LA_RED/MapServer'
LAYER_IMD   = '1025'   # IMD total de vehículos por tramo
PAGE_SIZE   = 1000     # ArcGIS default cap per request
RETRY_WAIT  = 3        # seconds between retries on transient errors
MAX_RETRIES = 3

JSON_RAW_PATH = os.path.join(RAW_DIR, 'imd_total_tramos_raw.json')

def fetch_arcgis_page(layer: str, offset: int, page_size: int) -> dict:
    """Fetch one page of features from the ArcGIS REST query endpoint."""
    params = urllib.parse.urlencode({
        'where':             'GEOM IS NOT NULL',
        'outFields':         '*',
        'returnGeometry':    'true',
        'resultOffset':      offset,
        'resultRecordCount': page_size,
        'f':                 'json',
    })
    url = f'{BASE_URL}/{layer}/query?{params}'
    req = urllib.request.Request(
        url, headers={'User-Agent': 'Mozilla/5.0 (IE-Datathon-2026)'}
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.loads(resp.read().decode('utf-8'))

def download_arcgis_layer(layer: str, dest_path: str) -> dict:
    """
    Download all features from an ArcGIS REST layer using pagination.
    Returns the merged FeatureCollection-like dict and saves raw JSON to dest_path.
    Skips download if file already exists.
    """
    if os.path.exists(dest_path):
        size_kb = os.path.getsize(dest_path) / 1024
        print(f'✅  {os.path.basename(dest_path)} already exists ({size_kb:.0f} KB) — loading from disk')
        with open(dest_path, 'r', encoding='utf-8') as f:
            return json.load(f)

    all_features = []
    offset       = 0
    page_num     = 1
    fields_meta  = None

    print(f'Downloading layer {layer} from ArcGIS REST...')
    while True:
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                data = fetch_arcgis_page(layer, offset, PAGE_SIZE)
                break
            except Exception as e:
                print(f'  Page {page_num}, attempt {attempt}/{MAX_RETRIES} failed: {e}')
                if attempt == MAX_RETRIES:
                    raise
                time.sleep(RETRY_WAIT)

        if 'error' in data:
            raise RuntimeError(f'ArcGIS error: {data["error"]}')

        features = data.get('features', [])
        all_features.extend(features)

        if fields_meta is None:
            fields_meta = data.get('fields', [])

        exceeded = data.get('exceededTransferLimit', False)
        print(f'  Page {page_num:3d} | offset {offset:6d} | '
              f'fetched {len(features):4d} | total so far {len(all_features):6d}'
              f'{" | more pages..." if exceeded else " | last page ✅"}')

        if not exceeded or len(features) == 0:
            break

        offset   += PAGE_SIZE
        page_num += 1
        time.sleep(0.2)  # polite delay

    # Assemble full response
    result = {
        'fields':            fields_meta,
        'geometryType':      data.get('geometryType', 'esriGeometryPolyline'),
        'spatialReference':  data.get('spatialReference', {'wkid': 4326}),
        'features':          all_features,
        '_total_features':   len(all_features),
        '_layer':            layer,
        '_source_url':       f'{BASE_URL}/{layer}/query',
    }

    with open(dest_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False)

    print(f'\n✅  Saved {len(all_features):,} features → {dest_path} '
          f'({os.path.getsize(dest_path)/1024/1024:.1f} MB)')
    return result

# ── Run download
raw_json = download_arcgis_layer(LAYER_IMD, JSON_RAW_PATH)
print(f'\nTotal features downloaded: {raw_json["_total_features"]:,}')
print(f'Fields available: {[f["name"] for f in raw_json["fields"]]}')

  Page   1 | offset      0 | fetched  568 | total so far    568 | last page ✅

✅  Saved 568 features → Data/Raw\imd_total_tramos_raw.json (2.1 MB)

Total features downloaded: 568
Fields available: ['id', 'Carretera', 'Longitud', 'PK_inicio', 'PK_fin', 'IMD_articulados', 'IMD_asc', 'IMD_autobuses', 'IMD_camionetas', 'IMD_coches', 'IMD_caravanas', 'IMD_desc', 'IMD_especiales', 'IMD_motos', 'IMD_semiremolques', 'IMD_total', 'IMD_tractores', 'IMD_trenes', 'Pesados', 'Dato_desde', 'Dato_hasta', 'Valido_desde', 'Valido_hasta', 'Titular', 'TENT', 'TENT_red_basica', 'TENT_corredor']


## 2. RAW Layer — Parse ArcGIS JSON and persist as GeoJSON

The ArcGIS REST response uses **Esri JSON** format, which differs from GeoJSON:  
- Geometry is stored as `{paths: [[[lon, lat], ...]]}` for polylines (not GeoJSON `coordinates`)  
- Attributes are nested under `feature.attributes`  
- Spatial reference is expressed as `{wkid: 4326}` (EPSG code)  

We convert to a GeoPandas GeoDataFrame and save a clean GeoJSON as the raw archive.

In [5]:
from shapely.geometry import shape, MultiLineString, LineString

def esri_polyline_to_shapely(geom: dict):
    """
    Convert an Esri polyline geometry dict to a Shapely geometry.
    Esri polylines use 'paths': list of rings, each a list of [x, y] coords.
    Returns LineString (single path) or MultiLineString (multiple paths).
    """
    if geom is None:
        return None
    paths = geom.get('paths', [])
    if not paths:
        return None
    lines = [LineString(path) for path in paths if len(path) >= 2]
    if not lines:
        return None
    return lines[0] if len(lines) == 1 else MultiLineString(lines)

def arcgis_json_to_gdf(raw: dict) -> gpd.GeoDataFrame:
    """
    Convert an ArcGIS REST JSON response dict to a GeoDataFrame.
    Handles both polyline and point geometry types.
    """
    features  = raw.get('features', [])
    geom_type = raw.get('geometryType', '')
    wkid      = raw.get('spatialReference', {}).get('wkid', 4326)

    records   = []
    geoms     = []

    for feat in features:
        attrs = feat.get('attributes', {})
        geom  = feat.get('geometry')

        if geom_type == 'esriGeometryPolyline':
            shapely_geom = esri_polyline_to_shapely(geom)
        elif geom_type == 'esriGeometryPoint':
            x, y = geom.get('x'), geom.get('y')
            from shapely.geometry import Point
            shapely_geom = Point(x, y) if x is not None else None
        else:
            # Fallback: try shapely shape() on GeoJSON-like dict
            try:
                shapely_geom = shape(geom) if geom else None
            except Exception:
                shapely_geom = None

        records.append(attrs)
        geoms.append(shapely_geom)

    df  = pd.DataFrame(records)
    gdf = gpd.GeoDataFrame(df, geometry=geoms, crs=f'EPSG:{wkid}')

    # Report geometry parse failures
    n_null_geom = gdf.geometry.isna().sum()
    if n_null_geom:
        print(f'⚠️  {n_null_geom} features had null/unparseable geometry — retained with null geom')

    print(f'GeoDataFrame created: {len(gdf):,} rows × {len(gdf.columns)} columns | CRS: {gdf.crs}')
    return gdf

print('arcgis_json_to_gdf() helper defined.')

arcgis_json_to_gdf() helper defined.


In [6]:
# ── Convert ArcGIS JSON → GeoDataFrame
print('='*60)
print('Parsing ArcGIS JSON → GeoDataFrame')
print('='*60)

gdf_total_raw = arcgis_json_to_gdf(raw_json)

print('\nShape:', gdf_total_raw.shape)
print('\nColumn dtypes:')
print(gdf_total_raw.dtypes)
print('\nFirst 3 rows (attributes only):')
gdf_total_raw.drop(columns='geometry').head(3)

Parsing ArcGIS JSON → GeoDataFrame
GeoDataFrame created: 568 rows × 28 columns | CRS: EPSG:25830

Shape: (568, 28)

Column dtypes:
id                      int64
Carretera              object
Longitud                int64
PK_inicio             float64
PK_fin                float64
IMD_articulados         int64
IMD_asc               float64
IMD_autobuses           int64
IMD_camionetas          int64
IMD_coches              int64
IMD_caravanas           int64
IMD_desc              float64
IMD_especiales          int64
IMD_motos               int64
IMD_semiremolques       int64
IMD_total               int64
IMD_tractores           int64
IMD_trenes              int64
Pesados               float64
Dato_desde              int64
Dato_hasta              int64
Valido_desde            int64
Valido_hasta           object
Titular                object
TENT                   object
TENT_red_basica        object
TENT_corredor          object
geometry             geometry
dtype: object

First 3 rows (

,id,Carretera,Longitud,PK_inicio,PK_fin,IMD_articulados,IMD_asc,IMD_autobuses,IMD_camionetas,IMD_coches,IMD_caravanas,IMD_desc,IMD_especiales,IMD_motos,IMD_semiremolques,IMD_total,IMD_tractores,IMD_trenes,Pesados,Dato_desde,Dato_hasta,Valido_desde,Valido_hasta,Titular,TENT,TENT_red_basica,TENT_corredor
0,270,A-31,753,165.54,166.30,1227,50.65,90,657,11530,45,49.35,5,49,370,14075,0,102,12.75,1640995200000,1672444800000,1717113600000,None,Estado,SI,Comprehensive,None
1,273,A-31,991,166.30,167.30,1227,50.65,90,657,11530,45,49.35,5,49,370,14075,0,102,12.75,1640995200000,1672444800000,1717113600000,None,Estado,SI,Comprehensive,None
2,276,A-31,2998,167.30,170.30,1227,50.65,90,657,11530,45,49.35,5,49,370,14075,0,102,12.75,1640995200000,1672444800000,1717113600000,None,Estado,SI,Comprehensive,None


In [7]:
# ── Save RAW GeoJSON (geometry + all original attributes, no transformation)
raw_geojson_path = os.path.join(RAW_DIR, 'imd_total_tramos_raw.geojson')

gdf_total_raw.to_file(raw_geojson_path, driver='GeoJSON')
print(f'Raw GeoJSON saved → {raw_geojson_path}')
print(f'File size: {os.path.getsize(raw_geojson_path)/1024/1024:.1f} MB')

Raw GeoJSON saved → Data/Raw\imd_total_tramos_raw.geojson
File size: 2.7 MB


### 3.1 Inspect raw attributes

ArcGIS JSON attributes map directly to columns — no HTML Description parsing needed.  
We inspect the field names and sample values to plan the renaming and cleaning steps.

In [8]:
# ── Field inventory from the ArcGIS metadata
print('Fields returned by ArcGIS REST service:')
for f in raw_json.get('fields', []):
    print(f'  {f["name"]:30s}  type={f["type"]:25s}  alias={f.get("alias", "")}')

print('\nGeoDataFrame columns:', gdf_total_raw.columns.tolist())
print('\nSample row:')
gdf_total_raw.drop(columns='geometry').iloc[0].to_frame('value')

Fields returned by ArcGIS REST service:
  id                              type=esriFieldTypeOID           alias=id
  Carretera                       type=esriFieldTypeString        alias=Carretera
  Longitud                        type=esriFieldTypeSingle        alias=Longitud
  PK_inicio                       type=esriFieldTypeDouble        alias=PK_inicio
  PK_fin                          type=esriFieldTypeDouble        alias=PK_fin
  IMD_articulados                 type=esriFieldTypeInteger       alias=IMD_articulados
  IMD_asc                         type=esriFieldTypeDouble        alias=IMD_asc
  IMD_autobuses                   type=esriFieldTypeInteger       alias=IMD_autobuses
  IMD_camionetas                  type=esriFieldTypeInteger       alias=IMD_camionetas
  IMD_coches                      type=esriFieldTypeInteger       alias=IMD_coches
  IMD_caravanas                   type=esriFieldTypeInteger       alias=IMD_caravanas
  IMD_desc                        type=esriFieldTyp

,value
id,270
Carretera,A-31
Longitud,753
PK_inicio,165.54
PK_fin,166.30
IMD_articulados,1227
IMD_asc,50.65
IMD_autobuses,90
IMD_camionetas,657
IMD_coches,11530


In [9]:
# ── No HTML Description parsing needed for ArcGIS JSON —
# attributes are already flat columns. Pass-through.
gdf_total = gdf_total_raw.copy()
print('Columns ready for renaming:', gdf_total.columns.tolist())

Columns ready for renaming: ['id', 'Carretera', 'Longitud', 'PK_inicio', 'PK_fin', 'IMD_articulados', 'IMD_asc', 'IMD_autobuses', 'IMD_camionetas', 'IMD_coches', 'IMD_caravanas', 'IMD_desc', 'IMD_especiales', 'IMD_motos', 'IMD_semiremolques', 'IMD_total', 'IMD_tractores', 'IMD_trenes', 'Pesados', 'Dato_desde', 'Dato_hasta', 'Valido_desde', 'Valido_hasta', 'Titular', 'TENT', 'TENT_red_basica', 'TENT_corredor', 'geometry']


## 3. PROCESSED Layer — Clean, Standardise & Filter

### 3.2 Column standardisation

ArcGIS field names from `mapas.fomento.gob.es` can be all-caps abbreviations.  
We apply a canonical rename map to produce clean, readable snake_case column names.

In [10]:
# ── Canonical rename map (keys = ArcGIS field names, values = clean names)
RENAME_MAP = {
    # Identifiers / road reference
    'CODRUTA':       'road_code',
    'CODTRAMO':      'segment_id',
    'DENOMINACION':  'road_name',
    'CARRETERA':     'road_code',
    'TRAMO':         'segment_id',
    'OBJECTID':      'object_id',
    'OBJECTID_1':    'object_id',
    # Chainage
    'PKINI':         'pk_start',
    'PKFIN':         'pk_end',
    'PK_INICIO':     'pk_start',
    'PK_FIN':        'pk_end',
    'LONGITUD':      'length_m',
    'LONGITUD_M':    'length_m',
    # Traffic
    'IMDTOT':        'imd_total',
    'IMD':           'imd_total',
    'IMD_TOTAL':     'imd_total',
    'IMDP':          'imd_heavy',
    'IMD_PESADOS':   'imd_heavy',
    'PCTPESADOS':    'pct_heavy',
    'PCT_PESADOS':   'pct_heavy',
    # Road classification
    'TIPO_VIA':      'road_type',
    'TIPO_CARRET':   'road_type',
    'TIPO_EST':      'station_type',
    'TIPOLOGIA':     'road_type',
    'TITULAR':       'owner',
    'TITULARIDAD':   'owner',
    # Year
    'AÑO':           'year',
    'ANO':           'year',
    'YEAR':          'year',
}

def apply_rename(gdf: gpd.GeoDataFrame, rename_map: dict) -> gpd.GeoDataFrame:
    upper_map    = {k.upper(): v for k, v in rename_map.items()}
    actual_rename = {c: upper_map[c.upper()] for c in gdf.columns
                     if c.upper() in upper_map}
    print(f'Renaming {len(actual_rename)} columns:')
    for old, new in actual_rename.items():
        print(f'  {old:25s} → {new}')
    unmapped = [c for c in gdf.columns
                if c != 'geometry' and c.upper() not in upper_map]
    if unmapped:
        print(f'\n  Unmapped columns (kept as-is): {unmapped}')
    return gdf.rename(columns=actual_rename)

gdf_total = apply_rename(gdf_total, RENAME_MAP)
print('\nColumns after rename:', gdf_total.columns.tolist())

Renaming 6 columns:
  Carretera                 → road_code
  Longitud                  → length_m
  PK_inicio                 → pk_start
  PK_fin                    → pk_end
  IMD_total                 → imd_total
  Titular                   → owner

  Unmapped columns (kept as-is): ['id', 'IMD_articulados', 'IMD_asc', 'IMD_autobuses', 'IMD_camionetas', 'IMD_coches', 'IMD_caravanas', 'IMD_desc', 'IMD_especiales', 'IMD_motos', 'IMD_semiremolques', 'IMD_tractores', 'IMD_trenes', 'Pesados', 'Dato_desde', 'Dato_hasta', 'Valido_desde', 'Valido_hasta', 'TENT', 'TENT_red_basica', 'TENT_corredor']

Columns after rename: ['id', 'road_code', 'length_m', 'pk_start', 'pk_end', 'IMD_articulados', 'IMD_asc', 'IMD_autobuses', 'IMD_camionetas', 'IMD_coches', 'IMD_caravanas', 'IMD_desc', 'IMD_especiales', 'IMD_motos', 'IMD_semiremolques', 'imd_total', 'IMD_tractores', 'IMD_trenes', 'Pesados', 'Dato_desde', 'Dato_hasta', 'Valido_desde', 'Valido_hasta', 'owner', 'TENT', 'TENT_red_basica', 'TENT_corredor

### 3.3 CRS Standardisation

ArcGIS REST returns coordinates in the spatial reference declared by the service.  
Layer 1025 uses **EPSG:4326** (WGS84). We verify and enforce this — the datathon output requires WGS84 decimal degrees.

In [11]:
TARGET_CRS = 'EPSG:4326'

def ensure_crs(gdf: gpd.GeoDataFrame, name: str) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        print(f'{name}: CRS is None — assigning {TARGET_CRS} (ArcGIS REST default)')
        gdf = gdf.set_crs(TARGET_CRS)
    elif gdf.crs.to_epsg() != 4326:
        print(f'{name}: CRS is {gdf.crs.to_string()} — reprojecting to {TARGET_CRS}')
        gdf = gdf.to_crs(TARGET_CRS)
    else:
        print(f'{name}: CRS is already {TARGET_CRS} ✅')
    return gdf

gdf_total = ensure_crs(gdf_total, 'IMD total')

IMD total: CRS is EPSG:25830 — reprojecting to EPSG:4326


### 3.4 Data type casting and numeric coercion

In [12]:
def coerce_numeric(gdf: gpd.GeoDataFrame, cols: list) -> gpd.GeoDataFrame:
    """Convert string columns to numeric, coercing errors to NaN.
    ArcGIS JSON already returns typed values, but some fields may arrive as
    strings if the service layer uses coded-value domains.
    """
    for col in cols:
        if col in gdf.columns:
            if gdf[col].dtype == object:
                # Handle Spanish decimal commas and thousands dots
                gdf[col] = gdf[col].astype(str).str.replace('.', '', regex=False)\
                                              .str.replace(',', '.', regex=False)\
                                              .str.strip()
            gdf[col] = pd.to_numeric(gdf[col], errors='coerce')
    return gdf

NUMERIC_COLS = ['imd_total', 'imd_heavy', 'pct_heavy', 'pk_start', 'pk_end', 'length_m', 'year']

gdf_total = coerce_numeric(gdf_total, NUMERIC_COLS)

print('Dtypes after coercion:')
print(gdf_total.dtypes)

Dtypes after coercion:
id                      int64
road_code              object
length_m                int64
pk_start              float64
pk_end                float64
IMD_articulados         int64
IMD_asc               float64
IMD_autobuses           int64
IMD_camionetas          int64
IMD_coches              int64
IMD_caravanas           int64
IMD_desc              float64
IMD_especiales          int64
IMD_motos               int64
IMD_semiremolques       int64
imd_total               int64
IMD_tractores           int64
IMD_trenes              int64
Pesados               float64
Dato_desde              int64
Dato_hasta              int64
Valido_desde            int64
Valido_hasta           object
owner                  object
TENT                   object
TENT_red_basica        object
TENT_corredor          object
geometry             geometry
dtype: object


### 3.5 Filter: keep only interurban roads (autopistas, autovías, carreteras nacionales)

**Assumption (documented per datathon rules):**  
Roads are classified as interurban if their `road_code` matches the official Ministry of Transport naming conventions:
- `AP-xx` → Autopista de peaje  
- `A-xx`  → Autovía del Estado  
- `N-xx`  → Carretera Nacional  
- `E-xx`  → Itinerario Europeo (E-roads, always interurban)

Codes starting with regional prefixes (e.g., `M-`, `B-`, `GI-`, `CV-`) are excluded as they correspond to regional/urban networks.

If a `road_type` column is present, we additionally filter on values indicating non-urban classification.

In [13]:
# ── Regex pattern for interurban road codes (RCE scope)
INTERURBAN_PATTERN = re.compile(
    r'^(AP|A|N|E)-?\d+',
    re.IGNORECASE
)

def filter_interurban(gdf: gpd.GeoDataFrame, name: str) -> gpd.GeoDataFrame:
    n_before = len(gdf)

    if 'road_code' in gdf.columns:
        mask = gdf['road_code'].astype(str).str.match(INTERURBAN_PATTERN)
        gdf = gdf[mask].copy()
        print(f'{name}: Filtered by road_code — {n_before:,} → {len(gdf):,} rows '
              f'(dropped {n_before - len(gdf):,} non-RCE segments)')
    else:
        print(f'{name}: No road_code column found — skipping code-based filter. '
              f'Verify columns: {gdf.columns.tolist()}')

    # Secondary filter on road_type if available
    if 'road_type' in gdf.columns:
        urban_types = ['urbano', 'urban', 'via urbana', 'ramal urbano']
        mask_type = ~gdf['road_type'].astype(str).str.lower().isin(urban_types)
        n_before2 = len(gdf)
        gdf = gdf[mask_type].copy()
        print(f'{name}: Filtered by road_type — {n_before2:,} → {len(gdf):,} rows')

    return gdf

gdf_total = filter_interurban(gdf_total, 'IMD total')

IMD total: Filtered by road_code — 568 → 506 rows (dropped 62 non-RCE segments)


### 3.6 Null value analysis and imputation strategy

In [14]:
def null_report(gdf: gpd.GeoDataFrame, name: str):
    nulls = gdf.isnull().sum()
    pct   = (nulls / len(gdf) * 100).round(1)
    report = pd.DataFrame({'null_count': nulls, 'null_pct': pct})
    report = report[report['null_count'] > 0].sort_values('null_pct', ascending=False)
    if report.empty:
        print(f'{name}: No nulls found ✅')
    else:
        print(f'{name}: Null summary:')
        print(report.to_string())
    return report

null_report(gdf_total, 'IMD total')

IMD total: Null summary:
                 null_count  null_pct
Valido_hasta            506    100.00
TENT_corredor           434     85.80
TENT_red_basica         342     67.60
IMD_asc                 282     55.70
IMD_desc                282     55.70
owner                     4      0.80


,null_count,null_pct
Valido_hasta,506,100.00
TENT_corredor,434,85.80
TENT_red_basica,342,67.60
IMD_asc,282,55.70
IMD_desc,282,55.70
owner,4,0.80


In [15]:
# ── Imputation strategy
#
# imd_total = NaN  → DROP (no demand value, segment unusable)
# pct_heavy = NaN  → IMPUTE median per road_type; fallback global median
#                    (only if the column exists — may be absent in layer 1025)
# year      = NaN  → IMPUTE mode
# All other NaNs   → KEEP (metadata, not used in calculations)

# 1. Drop rows where imd_total is missing
if 'imd_total' in gdf_total.columns:
    n_before = len(gdf_total)
    gdf_total = gdf_total.dropna(subset=['imd_total'])
    print(f'imd_total: dropped {n_before - len(gdf_total)} rows with null value')
else:
    print('⚠️  imd_total column not found — check rename map against actual field names above')

# 2. Impute pct_heavy (only if present at this stage)
if 'pct_heavy' in gdf_total.columns:
    if 'road_type' in gdf_total.columns:
        gdf_total['pct_heavy'] = gdf_total.groupby('road_type')['pct_heavy']\
                                          .transform(lambda x: x.fillna(x.median()))
    global_median_pct = gdf_total['pct_heavy'].median()
    gdf_total['pct_heavy'] = gdf_total['pct_heavy'].fillna(global_median_pct)
    remaining = gdf_total['pct_heavy'].isnull().sum()
    print(f'pct_heavy: nulls after imputation = {remaining} '
          f'(global median fallback = {global_median_pct:.1f}%)')
else:
    print('pct_heavy not present yet — will be derived in section 3.8')

# 3. Impute year with mode (if present)
if 'year' in gdf_total.columns:
    mode_year = int(gdf_total['year'].mode()[0]) if gdf_total['year'].notna().any() else None
    n_year_nulls = int(gdf_total['year'].isnull().sum())
    gdf_total['year'] = gdf_total['year'].fillna(mode_year)
    print(f'year: imputed {n_year_nulls} nulls with mode = {mode_year}')
else:
    print('year column not present — skipping year imputation')

print('\nNull check after imputation:')
null_report(gdf_total, 'IMD total')

imd_total: dropped 0 rows with null value
pct_heavy not present yet — will be derived in section 3.8
year column not present — skipping year imputation

Null check after imputation:
IMD total: Null summary:
                 null_count  null_pct
Valido_hasta            506    100.00
TENT_corredor           434     85.80
TENT_red_basica         342     67.60
IMD_asc                 282     55.70
IMD_desc                282     55.70
owner                     4      0.80


,null_count,null_pct
Valido_hasta,506,100.00
TENT_corredor,434,85.80
TENT_red_basica,342,67.60
IMD_asc,282,55.70
IMD_desc,282,55.70
owner,4,0.80


### 3.7 Duplicate detection and removal

Three types of duplicates are checked and handled separately:

| Type | Definition | Strategy |
|------|------------|----------|
| **Exact row duplicates** | All columns (including geometry) are identical | Drop, keep first |
| **Logical duplicates** | Same `segment_id` appears more than once (e.g. from multi-layer KML concatenation) | Keep row with highest `imd_total` (most recent/complete measurement) |
| **Spatial near-duplicates** | Different `segment_id` but overlapping geometry and same `road_code` | Flag only — not dropped, as they may represent bidirectional segments |

**Assumption (documented):** When a `segment_id` appears in multiple KML layers, the record with the highest `imd_total` is retained. This reflects the official HERMES methodology where later-added layers supersede earlier ones for the same physical segment.

In [16]:
# ────────────────────────────────────────────────
# TYPE 1: Exact row duplicates (all columns equal)
# ────────────────────────────────────────────────
def drop_exact_duplicates(gdf: gpd.GeoDataFrame, name: str) -> gpd.GeoDataFrame:
    # Geometry columns are not hashable for pandas duplicated() — compare via WKT
    non_geom_cols = [c for c in gdf.columns if c != 'geometry']
    gdf_check = gdf[non_geom_cols].copy()

    # Add WKT of geometry as a comparable column
    if gdf.geometry is not None:
        gdf_check['_geom_wkt'] = gdf.geometry.apply(
            lambda g: g.wkt if g is not None else None
        )

    mask_dup = gdf_check.duplicated(keep='first')
    n_exact = mask_dup.sum()

    if n_exact > 0:
        print(f'{name}: Dropping {n_exact} exact duplicate rows ({n_exact/len(gdf)*100:.2f}%)')
        gdf = gdf[~mask_dup].copy()
    else:
        print(f'{name}: No exact duplicate rows found ✅')

    return gdf

gdf_total = drop_exact_duplicates(gdf_total, 'IMD total')

IMD total: No exact duplicate rows found ✅


In [17]:
# ──────────────────────────────────────────────────────────────────────────
# TYPE 2: Logical duplicates — same segment_id, different rows
# Can arise from paginated ArcGIS responses if offset boundaries overlap
# ──────────────────────────────────────────────────────────────────────────
def drop_logical_duplicates(gdf: gpd.GeoDataFrame, id_col: str, value_col: str,
                             name: str) -> gpd.GeoDataFrame:
    if id_col not in gdf.columns:
        print(f'{name}: Column "{id_col}" not found — skipping logical dedup')
        return gdf

    n_before = len(gdf)
    duplicated_ids = gdf[gdf.duplicated(subset=[id_col], keep=False)][id_col].unique()
    n_dup_ids = len(duplicated_ids)

    if n_dup_ids == 0:
        print(f'{name}: No logical duplicates on "{id_col}" ✅')
        return gdf

    print(f'{name}: {n_dup_ids} segment_ids appear more than once → '
          f'keeping row with highest "{value_col}"')

    if value_col in gdf.columns:
        gdf = gdf.sort_values(value_col, ascending=False)

    gdf = gdf.drop_duplicates(subset=[id_col], keep='first')
    print(f'{name}: {n_before:,} → {len(gdf):,} rows after logical dedup '
          f'(removed {n_before - len(gdf):,})')
    return gpd.GeoDataFrame(gdf, geometry='geometry', crs=gdf.crs)

gdf_total = drop_logical_duplicates(gdf_total, id_col='segment_id',
                                     value_col='imd_total', name='IMD total')

IMD total: Column "segment_id" not found — skipping logical dedup


In [18]:
# ──────────────────────────────────────────────────────────────────────────
# TYPE 3: Spatial near-duplicates — same road_code, overlapping geometry
# Flag only (do NOT drop): bidirectional lanes share >90% geometry by design
# ──────────────────────────────────────────────────────────────────────────
def flag_spatial_near_duplicates(gdf: gpd.GeoDataFrame, name: str,
                                  road_col: str = 'road_code') -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    gdf['spatial_dup_flag'] = False

    if road_col not in gdf.columns:
        print(f'{name}: Column "{road_col}" not found — skipping spatial near-dup check')
        return gdf

    flagged = 0
    for road, group in gdf.groupby(road_col):
        if len(group) < 2:
            continue
        geoms = group.geometry.values
        idx   = group.index.tolist()
        for i in range(len(geoms)):
            for j in range(i + 1, len(geoms)):
                if geoms[i] is None or geoms[j] is None:
                    continue
                try:
                    overlap  = geoms[i].intersection(geoms[j]).length
                    min_len  = min(geoms[i].length, geoms[j].length)
                    if min_len > 0 and (overlap / min_len) > 0.90:
                        gdf.loc[idx[j], 'spatial_dup_flag'] = True
                        flagged += 1
                except Exception:
                    pass

    print(f'{name}: Spatial near-duplicate flag — {flagged} segments '
          f'({flagged/len(gdf)*100:.2f}%) [flagged, not dropped]')
    return gdf

gdf_total = flag_spatial_near_duplicates(gdf_total, 'IMD total')

print('\nSpatial dup flag distribution:')
print(gdf_total['spatial_dup_flag'].value_counts())

IMD total: Spatial near-duplicate flag — 0 segments (0.00%) [flagged, not dropped]

Spatial dup flag distribution:
spatial_dup_flag
False    506
Name: count, dtype: int64


In [19]:
# ── Deduplication summary
print('='*60)
print('DEDUPLICATION SUMMARY')
print('='*60)
print(f'IMD total — final row count: {len(gdf_total):,}')

if 'spatial_dup_flag' in gdf_total.columns:
    n_spatial = gdf_total['spatial_dup_flag'].sum()
    print(f'Spatial near-duplicates flagged: {n_spatial}')
    if n_spatial > 0:
        print('  → Inspect: gdf_total[gdf_total.spatial_dup_flag == True]')

print('\n✅  Deduplication complete.')

DEDUPLICATION SUMMARY
IMD total — final row count: 506
Spatial near-duplicates flagged: 0

✅  Deduplication complete.


### 3.8 Derived columns

Add calculated fields useful for the charging demand model:
- `imd_light` — light vehicle IMD (total minus heavy)
- `centroid_lat` / `centroid_lon` — representative point for each segment (for spatial joins with grid capacity data)
- `length_km` — segment length in km
- `demand_index` — normalised demand proxy = `imd_total × length_km` (vehicle-km per day)

In [20]:
# ── Diagnostic: show actual columns available after cleaning
print('Columns available for derived fields:')
print([c for c in gdf_total.columns if c != 'geometry'])

# ── imd_light  (light vehicles = total − heavy)
# Only computed if both source columns are present;
# layer 1025 may not include the heavy-vehicle breakdown.
if 'imd_total' in gdf_total.columns and 'imd_heavy' in gdf_total.columns:
    gdf_total['imd_light'] = (gdf_total['imd_total'] - gdf_total['imd_heavy']).clip(lower=0)
    print('imd_light: computed from imd_total - imd_heavy')
elif 'imd_total' in gdf_total.columns and 'pct_heavy' in gdf_total.columns:
    gdf_total['imd_light'] = (gdf_total['imd_total'] * (1 - gdf_total['pct_heavy'] / 100)).round(0)
    print('imd_light: computed from imd_total × (1 - pct_heavy/100)')
else:
    # ASSUMPTION: no heavy-vehicle split available in this layer.
    # Industry average for Spanish interurban roads: ~15% heavy vehicles
    # Source: Anuario Estadístico de Carreteras 2024 (MITMS), Table 1.2.3
    PCT_HEAVY_DEFAULT = 15.0
    gdf_total['imd_heavy'] = (gdf_total['imd_total'] * PCT_HEAVY_DEFAULT / 100).round(0)
    gdf_total['pct_heavy'] = PCT_HEAVY_DEFAULT
    gdf_total['imd_light'] = (gdf_total['imd_total'] - gdf_total['imd_heavy']).clip(lower=0)
    print(f'imd_light: estimated using default pct_heavy={PCT_HEAVY_DEFAULT}% '
          f'(national average, MITMS 2024) — flagged as assumption')

# ── length_km
if 'length_m' in gdf_total.columns:
    gdf_total['length_km'] = (gdf_total['length_m'] / 1000).round(3)
    print('length_km: from length_m field')
else:
    # Compute from geometry — reproject to ETRS89/UTM zone 30N (EPSG:25830)
    # for accurate metric distances in mainland Spain
    gdf_metric = gdf_total.to_crs('EPSG:25830')
    gdf_total['length_km'] = (gdf_metric.geometry.length / 1000).round(3)
    print('length_km: computed from geometry (ETRS89/UTM zone 30N, EPSG:25830)')

# ── demand_index  (vehicle-km / day — standard traffic planning metric)
gdf_total['demand_index'] = (gdf_total['imd_total'] * gdf_total['length_km']).round(0)

# ── Centroid coordinates for spatial joins with grid capacity data
gdf_total['centroid_lon'] = gdf_total.geometry.centroid.x.round(6)
gdf_total['centroid_lat'] = gdf_total.geometry.centroid.y.round(6)

# ── Preview — only show columns that actually exist
preview_cols = ['road_code', 'imd_total', 'imd_light', 'imd_heavy', 'pct_heavy',
                'length_km', 'demand_index', 'centroid_lat', 'centroid_lon']
available_preview = [c for c in preview_cols if c in gdf_total.columns]
print(f'\nDerived columns added: {[c for c in ["imd_light","imd_heavy","pct_heavy","length_km","demand_index","centroid_lat","centroid_lon"] if c in gdf_total.columns]}')
gdf_total[available_preview].head(5)

Columns available for derived fields:
['id', 'road_code', 'length_m', 'pk_start', 'pk_end', 'IMD_articulados', 'IMD_asc', 'IMD_autobuses', 'IMD_camionetas', 'IMD_coches', 'IMD_caravanas', 'IMD_desc', 'IMD_especiales', 'IMD_motos', 'IMD_semiremolques', 'imd_total', 'IMD_tractores', 'IMD_trenes', 'Pesados', 'Dato_desde', 'Dato_hasta', 'Valido_desde', 'Valido_hasta', 'owner', 'TENT', 'TENT_red_basica', 'TENT_corredor', 'spatial_dup_flag']
imd_light: estimated using default pct_heavy=15.0% (national average, MITMS 2024) — flagged as assumption
length_km: from length_m field

Derived columns added: ['imd_light', 'imd_heavy', 'pct_heavy', 'length_km', 'demand_index', 'centroid_lat', 'centroid_lon']


C:\Users\alber\AppData\Local\Temp\ipykernel_4980\1777878418.py:40: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_total['centroid_lon'] = gdf_total.geometry.centroid.x.round(6)
C:\Users\alber\AppData\Local\Temp\ipykernel_4980\1777878418.py:41: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_total['centroid_lat'] = gdf_total.geometry.centroid.y.round(6)


,road_code,imd_total,imd_light,imd_heavy,pct_heavy,length_km,demand_index,centroid_lat,centroid_lon
0,A-31,14075,11964.00,2111.00,15.00,0.75,10598.00,38.77,-0.96
1,A-31,14075,11964.00,2111.00,15.00,0.99,13948.00,38.76,-0.96
2,A-31,14075,11964.00,2111.00,15.00,3.00,42197.00,38.75,-0.95
3,A-31,26611,22619.00,3992.00,15.00,0.53,14024.00,38.73,-0.94
4,A-31,26611,22619.00,3992.00,15.00,1.97,52344.00,38.72,-0.94


### 3.9 Assign clean output variable

Single-source pipeline: `gdf_total` is the final cleaned GeoDataFrame.  
We alias it as `gdf_clean` for clarity in the steps that follow.

In [21]:
gdf_merged = gdf_total.copy()
print(f'Clean GeoDataFrame ready: {len(gdf_merged):,} rows × {len(gdf_merged.columns)} columns')

Clean GeoDataFrame ready: 506 rows × 36 columns


### 3.10 Outlier detection and flagging

We flag extreme IMD values using IQR-based bounds (1.5×IQR) separately per `road_type`.  
Outliers are **flagged, not removed**, to preserve data integrity for the optimisation model.

In [22]:
def flag_outliers_iqr(gdf: gpd.GeoDataFrame, col: str, group_col: str = None) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    flag_col = f'{col}_outlier_flag'

    if group_col and group_col in gdf.columns:
        def iqr_bounds(x):
            q1, q3 = x.quantile(0.25), x.quantile(0.75)
            iqr = q3 - q1
            return (x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)
        gdf[flag_col] = gdf.groupby(group_col)[col].transform(iqr_bounds)
    else:
        q1, q3 = gdf[col].quantile(0.25), gdf[col].quantile(0.75)
        iqr = q3 - q1
        gdf[flag_col] = (gdf[col] < q1 - 1.5 * iqr) | (gdf[col] > q3 + 1.5 * iqr)

    n_flags = int(gdf[flag_col].sum())
    pct = n_flags / len(gdf) * 100
    print(f'Outlier flags on "{col}" (grouped by {group_col}): {n_flags} ({pct:.1f}%) — flagged, not dropped')
    return gdf

if 'imd_total' in gdf_merged.columns:
    gdf_merged = flag_outliers_iqr(gdf_merged, 'imd_total', group_col='road_type')

print('\nOutlier flag value counts:')
if 'imd_total_outlier_flag' in gdf_merged.columns:
    print(gdf_merged['imd_total_outlier_flag'].value_counts())

Outlier flags on "imd_total" (grouped by road_type): 37 (7.3%) — flagged, not dropped

Outlier flag value counts:
imd_total_outlier_flag
False    469
True      37
Name: count, dtype: int64


### 3.11 Final column selection and ordering

In [23]:
FINAL_COLS_ORDERED = [
    # Identifiers
    'segment_id', 'road_code', 'road_name',
    # Chainage
    'pk_start', 'pk_end', 'length_m', 'length_km',
    # Road metadata
    'road_type', 'station_type', 'owner', 'year',
    # IMD values
    'imd_total', 'imd_light', 'imd_heavy', 'pct_heavy',
    # Derived demand
    'demand_index', 'centroid_lat', 'centroid_lon',
    # Flags
    'imd_total_outlier_flag', 'spatial_dup_flag',
    # Geometry
    'geometry'
]

final_cols = [c for c in FINAL_COLS_ORDERED if c in gdf_merged.columns]
gdf_clean = gdf_merged[final_cols].copy()

print(f'Final column count : {len(final_cols)}')
print(f'Final row count    : {len(gdf_clean):,}')
print('\nFinal columns:')
print(final_cols)

Final column count : 16
Final row count    : 506

Final columns:
['road_code', 'pk_start', 'pk_end', 'length_m', 'length_km', 'owner', 'imd_total', 'imd_light', 'imd_heavy', 'pct_heavy', 'demand_index', 'centroid_lat', 'centroid_lon', 'imd_total_outlier_flag', 'spatial_dup_flag', 'geometry']


## 4. Save Processed Outputs

In [24]:
# ── Save as GeoJSON (full dataset with geometry)
processed_geojson = os.path.join(PROCESSED_DIR, 'imd_interurban_cleaned.geojson')
gdf_clean.to_file(processed_geojson, driver='GeoJSON')
print(f'GeoJSON saved → {processed_geojson}')

GeoJSON saved → Data/Processed\imd_interurban_cleaned.geojson


In [25]:
# ── Save as CSV (without geometry, for tabular analysis)
processed_csv = os.path.join(PROCESSED_DIR, 'imd_interurban_cleaned.csv')
df_csv = gdf_clean.drop(columns=['geometry']).copy()
df_csv.to_csv(processed_csv, index=False, encoding='utf-8')
print(f'CSV saved → {processed_csv}')

CSV saved → Data/Processed\imd_interurban_cleaned.csv


## 5. Output Verification

Mandatory checks per datathon eligibility rules.

In [26]:
# ── Reload and verify structure
df_check = pd.read_csv(processed_csv)

print('='*60)
print('OUTPUT VERIFICATION — imd_interurban_cleaned.csv')
print('='*60)
print(f'Shape  : {df_check.shape}')
print(f'Columns: {list(df_check.columns)}')
print('\nDtypes:')
print(df_check.dtypes)
print('\nDescriptive statistics (key numeric cols):')
key_cols = [c for c in ['imd_total', 'imd_light', 'imd_heavy', 'pct_heavy', 'length_km', 'demand_index'] if c in df_check.columns]
print(df_check[key_cols].describe().round(1))
print('\nNull count per column:')
print(df_check.isnull().sum()[df_check.isnull().sum() > 0])

OUTPUT VERIFICATION — imd_interurban_cleaned.csv
Shape  : (506, 15)
Columns: ['road_code', 'pk_start', 'pk_end', 'length_m', 'length_km', 'owner', 'imd_total', 'imd_light', 'imd_heavy', 'pct_heavy', 'demand_index', 'centroid_lat', 'centroid_lon', 'imd_total_outlier_flag', 'spatial_dup_flag']

Dtypes:
road_code                  object
pk_start                  float64
pk_end                    float64
length_m                    int64
length_km                 float64
owner                      object
imd_total                   int64
imd_light                 float64
imd_heavy                 float64
pct_heavy                 float64
demand_index              float64
centroid_lat              float64
centroid_lon              float64
imd_total_outlier_flag       bool
spatial_dup_flag             bool
dtype: object

Descriptive statistics (key numeric cols):
       imd_total  imd_light  imd_heavy  pct_heavy  length_km  demand_index
count     506.00     506.00     506.00     506.00     5

In [27]:
# ── Verify interurban road coverage
if 'road_code' in df_check.columns:
    print('Road codes present (sample):')
    print(df_check['road_code'].value_counts().head(20))

if 'road_type' in df_check.columns:
    print('\nRoad type distribution:')
    print(df_check['road_type'].value_counts())

if 'year' in df_check.columns:
    print('\nYear distribution:')
    print(df_check['year'].value_counts().sort_index())

Road codes present (sample):
road_code
A-4      29
A-66     26
N-340    26
N-634    22
N-6      17
A-33     16
N-332    16
N-430    14
N-401    14
A-3      13
N-260    13
A-6      12
A-5      12
N-420    12
A-54     10
N-502    10
N-110    10
N-630     9
A-2       9
A-31      8
Name: count, dtype: int64


In [28]:
# ── Sanity check: IMD ranges
# ASSUMPTION: IMD on interurban RCE roads reasonably ranges from ~100 to ~200,000 veh/day
# Source: Anuario Estadístico de Carreteras 2024 (MITMS)
if 'imd_total' in df_check.columns:
    imd_min = df_check['imd_total'].min()
    imd_max = df_check['imd_total'].max()
    imd_median = df_check['imd_total'].median()
    print(f'IMD total — min: {imd_min:,.0f} | median: {imd_median:,.0f} | max: {imd_max:,.0f} veh/day')
    if imd_max > 250000:
        print('⚠️  WARNING: Unusually high IMD values detected — check for unit errors in source data')
    else:
        print('✅  IMD range is within expected bounds')

print('\n✅  Verification complete.')

IMD total — min: 77 | median: 8,116 | max: 161,620 veh/day
✅  IMD range is within expected bounds

✅  Verification complete.


## 6. Summary, Assumptions & Limitations

### Outputs

| File | Format | Content |
|------|--------|---------|
| `Data/Raw/imd_total_tramos_raw.geojson` | GeoJSON | Raw IMD total dataset, unmodified |
| `Data/Raw/imd_tipo_vehiculo_tramos_raw.geojson` | GeoJSON | Raw IMD by vehicle type, unmodified |
| `Data/Processed/imd_interurban_cleaned.geojson` | GeoJSON | Clean, filtered, merged — with geometry |
| `Data/Processed/imd_interurban_cleaned.csv` | CSV | Clean, filtered, merged — tabular only |

### Documented Assumptions

1. **Interurban filter:** Roads are classified as interurban if their code matches `AP-xx`, `A-xx`, `N-xx`, or `E-xx` per the official Ministry of Transport naming convention (Real Decreto 1812/1994 — Reglamento General de Carreteras). All other prefixes correspond to regional or municipal networks and are excluded.

2. **Missing `pct_heavy`:** Imputed using the median value within each `road_type` group. This is conservative and consistent with the official methodology described in the Anuario Estadístico de Carreteras 2024 (MITMS).

3. **Missing `year`:** Imputed with the statistical mode across the dataset. If the dataset corresponds to a single year (as is typical in HERMES releases), this affects no records.

4. **Geometry length:** When `length_m` is absent, computed from the `geometry` column after reprojection to ETRS89/UTM zone 30N (EPSG:25830), which is the official Spanish reference system for metric calculations.

5. **Deduplication — logical duplicates:** When a `segment_id` appears in multiple KML layers (result of multi-layer concatenation in `read_kmz`), the record with the highest `imd_total` is retained. This reflects the HERMES convention where higher-traffic measurements supersede lower ones for the same physical segment.  

6. **Deduplication — spatial near-duplicates:** Segments sharing >90% of their geometry on the same road are flagged (`spatial_dup_flag = True`) but not removed, as they may represent bidirectional lanes or overlapping measurement intervals. They should be inspected before use in spatial joins.  

7. **Outlier handling:** IMD outliers flagged via IQR×1.5 per road type, not removed. High-IMD segments near urban access points may appear as statistical outliers but represent genuine peak demand — keeping them ensures the model captures the highest-priority charging corridors.

8. **`demand_index`:** Defined as `imd_total × length_km` (vehicle-km per day), a standard traffic planning metric for weighting infrastructure needs along a corridor. Reference: MITMS Plan de Aforos methodology.

### Limitations
- HERMES IMD data is updated annually; the version year should be verified at download time.
- Toll motorway virtual stations (282 stations) rely on electronic counting rather than physical inductive loops — they are included but may have different accuracy characteristics.
- Estimated (`Est`) station types use projected values where direct measurement is unavailable; these are flagged via `station_type` and should be treated with lower confidence in the optimisation model.

In [29]:
# ──────────────────────────────────────────────────────────────
# 4.4  Parquet — tabular data (sin geometría)
#      Para uso interno del modelo: más rápido y ligero que CSV,
#      preserva dtypes exactos sin inferencia al leer.
# ──────────────────────────────────────────────────────────────
# !pip install pyarrow --quiet  # descomenta si no está instalado

processed_parquet = os.path.join(PROCESSED_DIR, 'imd_interurban_cleaned.parquet')

# df_csv ya está definido en la celda anterior (gdf_clean sin geometry)
df_csv.to_parquet(
    processed_parquet,
    index       = False,
    engine      = 'pyarrow',
    compression = 'snappy',   # balance óptimo velocidad / tamaño
)

size_kb = os.path.getsize(processed_parquet) / 1024
print(f'✅  Parquet saved → {processed_parquet}')
print(f'    Rows: {len(df_csv):,} | Columns: {len(df_csv.columns)} | Size: {size_kb:.0f} KB')

# Verificación — recargar y confirmar que dtypes se preservan
df_parquet_check = pd.read_parquet(processed_parquet)
print(f'\nVerificación tras recarga:')
print(f'  Shape : {df_parquet_check.shape}')
print(f'  Dtypes:')
print(df_parquet_check.dtypes.to_string())

✅  Parquet saved → Data/Processed\imd_interurban_cleaned.parquet
    Rows: 506 | Columns: 15 | Size: 37 KB

Verificación tras recarga:
  Shape : (506, 15)
  Dtypes:
road_code                  object
pk_start                  float64
pk_end                    float64
length_m                    int64
length_km                 float64
owner                      object
imd_total                   int64
imd_light                 float64
imd_heavy                 float64
pct_heavy                 float64
demand_index              float64
centroid_lat              float64
centroid_lon              float64
imd_total_outlier_flag       bool
spatial_dup_flag             bool


In [30]:
# ──────────────────────────────────────────────────────────────
# 4.5  GeoParquet — geometría + atributos
#      Para spatial joins con datos de red eléctrica (i-DE,
#      Endesa, Viesgo). Requiere geopandas >= 0.12 y pyarrow.
# ──────────────────────────────────────────────────────────────
processed_geoparquet = os.path.join(PROCESSED_DIR, 'imd_interurban_cleaned.geoparquet')

try:
    gdf_clean.to_parquet(processed_geoparquet, compression='snappy')
    size_mb = os.path.getsize(processed_geoparquet) / 1024 / 1024
    print(f'✅  GeoParquet saved → {processed_geoparquet}')
    print(f'    Rows: {len(gdf_clean):,} | CRS: {gdf_clean.crs} | Size: {size_mb:.1f} MB')

    # Verificación — recargar y confirmar geometría intacta
    gdf_parquet_check = gpd.read_parquet(processed_geoparquet)
    print(f'\nVerificación tras recarga:')
    print(f'  Shape    : {gdf_parquet_check.shape}')
    print(f'  CRS      : {gdf_parquet_check.crs}')
    print(f'  Geom type: {gdf_parquet_check.geometry.geom_type.value_counts().to_dict()}')

except Exception as e:
    print(f'⚠️  GeoParquet no disponible: {e}')
    print('    Se mantiene GeoJSON como formato espacial de referencia.')
    processed_geoparquet = processed_geojson

✅  GeoParquet saved → Data/Processed\imd_interurban_cleaned.geoparquet
    Rows: 506 | CRS: EPSG:4326 | Size: 0.8 MB

Verificación tras recarga:
  Shape    : (506, 16)
  CRS      : {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "u